# 01 - Data Cleaning

**Dataset:** Telco Customer Churn (WA_Fn-UseC_-Telco-Customer-Churn.csv)

**Mục tiêu:**
1. Đọc dữ liệu thô
2. Sửa lỗi khoảng trắng cột `TotalCharges`
3. Xử lý giá trị thiếu (missing values)
4. Lọc trùng lặp
5. Ép kiểu dữ liệu
6. Lưu dataset sạch vào `data/processed/clean_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from data_loader import (
    load_raw_data, fix_total_charges, remove_duplicates,
    cast_dtypes, clean_data, save_clean_data,
    get_data_dictionary, get_data_summary,
    CATEGORICAL_COLUMNS, NUMERIC_COLUMNS,
)

## 1. Đọc dữ liệu thô

In [ ]:
raw_df = load_raw_data()
print(f"Shape: {raw_df.shape}")
raw_df.head()

In [ ]:
raw_df.info()

In [ ]:
raw_df.describe(include='all')

## 2. Data Dictionary

Bảng giải thích chi tiết ý nghĩa từng trường dữ liệu trong dataset.

In [ ]:
data_dict = get_data_dictionary()
data_dict.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

## 3. Kiểm tra dữ liệu trước khi làm sạch

### 3.1 Kiểm tra giá trị thiếu (Missing Values)

In [ ]:
print("Null values per column:")
print(raw_df.isnull().sum())
print(f"\nTổng null: {raw_df.isnull().sum().sum()}")

### 3.2 Kiểm tra cột TotalCharges

Cột `TotalCharges` đang ở kiểu `object` (string) thay vì `float`. Nguyên nhân: có 11 dòng chứa chuỗi rỗng (khoảng trắng) — tất cả đều có `tenure = 0` (khách hàng mới đăng ký, chưa phát sinh chi phí).

In [ ]:
print(f"Kiểu dữ liệu TotalCharges: {raw_df['TotalCharges'].dtype}")

empty_mask = raw_df['TotalCharges'].str.strip() == ''
empty_rows = raw_df[empty_mask]
print(f"\nSố dòng TotalCharges rỗng (khoảng trắng): {len(empty_rows)}")
empty_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

### 3.3 Kiểm tra trùng lặp

In [ ]:
print(f"Số dòng trùng lặp (toàn bộ cột): {raw_df.duplicated().sum()}")
print(f"Số customerID trùng lặp: {raw_df['customerID'].duplicated().sum()}")

### 3.4 Kiểm tra kiểu dữ liệu

In [ ]:
get_data_summary(raw_df)

## 4. Làm sạch dữ liệu

### 4.1 Sửa lỗi cột TotalCharges

- Loại bỏ khoảng trắng thừa
- Chuyển chuỗi rỗng thành `NaN`
- Ép sang kiểu `float`
- Điền `NaN` bằng giá trị `MonthlyCharges` (vì tenure = 0, tổng phí = phí tháng đầu)

In [ ]:
df = fix_total_charges(raw_df)
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"TotalCharges null: {df['TotalCharges'].isnull().sum()}")

prev_empty_ids = empty_rows['customerID'].values
df[df['customerID'].isin(prev_empty_ids)][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

### 4.2 Lọc trùng lặp

In [ ]:
df = remove_duplicates(df)
print(f"Shape sau lọc trùng lặp: {df.shape}")

### 4.3 Ép kiểu dữ liệu

In [ ]:
df = cast_dtypes(df)
df = df.reset_index(drop=True)
print("Kiểu dữ liệu sau khi ép:")
print(df.dtypes)

## 5. Kiểm tra dữ liệu sau khi làm sạch

In [ ]:
print(f"Shape: {df.shape}")
print(f"Null: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
get_data_summary(df)

In [ ]:
df.describe()

In [ ]:
df.head(10)

## 6. Pipeline hoàn chỉnh

Chạy toàn bộ pipeline từ đầu và lưu kết quả.

In [ ]:
clean_df = clean_data(raw_df)
output_path = save_clean_data(clean_df)
print(f"Dataset sạch đã lưu tại: {output_path}")
print(f"Shape: {clean_df.shape}")
clean_df.head()

## 7. Tóm tắt

| Bước | Mô tả | Kết quả |
|------|-------|---------|
| 1 | Đọc dữ liệu thô | 7043 dòng, 21 cột |
| 2 | Sửa TotalCharges (khoảng trắng → NaN → MonthlyCharges) | 11 dòng được sửa |
| 3 | Lọc trùng lặp | 0 dòng bị xóa |
| 4 | Ép kiểu (TotalCharges → float, categorical columns) | Hoàn tất |
| 5 | Lưu dataset sạch | `data/processed/clean_data.csv` |